In [ ]:
# print("Hello world")

Hello world


In [1]:
q1 = "I just discovered the course, can I still join?"
q2 = "I just found out about the program, can I still enroll?"

In [2]:
!pip install -U sentence-transformers


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

c:\Users\RickyS-PC\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7821.65it/s]


In [4]:
!pip install minsearch


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [3]:
from ingest import load_faq_data, build_index

In [6]:
!pip install python-dotenv


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [6]:
!pip install google-genai

  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
   ---------------------------------------- 0.0/958.0 kB ? eta -:--:--
   --------------------- ------------------ 524.3/958.0 kB 3.1 MB/s eta 0:00:01
   -------------------------------- ------- 786.4/958.0 kB 3.2 MB/s eta 0:00:01
   ---------------------------------------- 958.0/958.0 kB 1.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.1 MB 3.3 MB/s eta 0:00:01
   ------------------------- -------------- 1.3/2.1 MB 3.9 MB/s eta 0:00:01
   ----------------------------------- ---- 1.8/2.1 MB 3.4 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 3.5 MB/s eta 0:00:00
Using cached pyasn1_modules-0.4.2-py3-none-any.whl (181 kB)

  Attempting uninstall: pydantic-core

    Found existing installation: pydantic_core 2.41.4

    Uninstalling pydantic_core-2.41.4:

      Successfully uninstalled pydantic_core-2.41.

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
apache-airflow-task-sdk 1.1.1 requires psutil>=6.1.0, but you have psutil 5.9.0 which is incompatible.


In [7]:
import os
import json
from minsearch import Index
from dotenv import load_dotenv
load_dotenv()

True

In [8]:
from google import genai
from google.genai import types
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

In [9]:
v1 = model.encode(q1)

In [9]:
v1.shape

(384,)

In [10]:
v2 = model.encode(q2)

In [11]:
d = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."

In [12]:
dv = model.encode(d)

In [14]:
v1.dot(dv)

np.float32(0.39572883)

In [15]:
v2.dot(dv)

np.float32(0.46365508)

In [13]:
documents = load_faq_data()

In [15]:
print(documents[10])

{'id': '316180784f', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: How many hours per week am I expected to spend on this course?', 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}


In [14]:
json_object = json.dumps(documents[10], indent=4)

In [17]:
print(type(json_object))

<class 'str'>


In [18]:
print(json_object)

{
    "id": "316180784f",
    "course": "data-engineering-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Course: How many hours per week am I expected to spend on this course?",
    "answer": "It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer."
}


In [15]:
texts = []

for doc in documents:
    text = doc['question'] + ' ' + doc['answer']
    texts.append(text)

In [20]:
print(len(texts))

1368


In [16]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

100%|██████████| 28/28 [00:30<00:00,  1.08s/it]


In [17]:
v1.dot(vectors[10])

np.float32(0.31346333)

In [18]:
scores = []

for i in range(len(vectors)):
    score = v1.dot(vectors[i])
    scores.append(score)

In [19]:
import numpy as np
X = np.array(vectors)

In [20]:
scores = X.dot(v1)

In [21]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(538), np.float32(0.831779))

In [27]:
documents[553]

{'id': 'a9353fadfe',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'The homework submission form is still open even though the deadline has passed — can I still submit?',
 'answer': "Yes. As long as the submission form is still open, you can submit your answers, even if the listed deadline has already passed. You can no longer submit only after the form has been closed — so while it's still open, go ahead and submit."}

In [22]:
top5 = np.argsort(scores)[-5:]
top5 = top5[::-1]

In [29]:
scores[top5]

array([0.831779  , 0.6845695 , 0.61755615, 0.6088087 , 0.58479655],
      dtype=float32)

In [23]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.831779
{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

0.68456936
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'The course has already started. Can I still join it?', 'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'}

0.6175562
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'se

In [24]:
top5 = np.argsort(-scores)[:5]

In [32]:
print(top5)

[538 925 643   2 503]


In [25]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X, documents)

In [25]:
vindex.search(v1, num_results=5, filter_dict={'course': 'llm-zoomcamp'})

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the co

In [26]:
index = build_index(documents)

In [27]:
from rag_helper import RAGBase

assistant = RAGBase(index, client)

In [28]:
query = 'I just found out about the program, can I still sign up?'
assistant.rag(query)

'Yes, you can still join. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

In [29]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {'course': self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [30]:
!pip install sqlitesearch

   ---------------------------------------- 0.0/2.8 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.8 MB ? eta -:--:--
   ----------- ---------------------------- 0.8/2.8 MB 2.1 MB/s eta 0:00:01
   -------------- ------------------------- 1.0/2.8 MB 2.0 MB/s eta 0:00:01
   ---------------------- ----------------- 1.6/2.8 MB 2.1 MB/s eta 0:00:01
   -------------------------- ------------- 1.8/2.8 MB 2.1 MB/s eta 0:00:01
   ------------------------------------- -- 2.6/2.8 MB 2.1 MB/s eta 0:00:01
   ---------------------------------------- 2.8/2.8 MB 2.0 MB/s eta 0:00:00
   ---------------------------------------- 0.0/41.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/41.9 MB ? eta -:--:--
    --------------------------------------- 0.5/41.9 MB 5.4 MB/s eta 0:00:08
    --------------------------------------- 0.5/41.9 MB 5.4 MB/s eta 0:00:08
    --------------------------------------- 0.8/41.9 MB 1.2 MB/s eta 0:00:36
   - ------------------------------

In [33]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=['course'],
    mode='ivf',
    db_path='faq_vectors2.db'
)

In [34]:
vs_index.fit(vectors, documents)

In [35]:
query = 'I just discovered the course. Can I still join it?'
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [37]:
# results

In [38]:
results = vs_index.search(
    query_vector,
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

In [39]:
vs_index.close()